1

In [ ]:

import os
import re
import numpy as np
import pandas as pd
from scipy.io import loadmat


def _make_range(start_id, class_name, fault_size, fault_position, fs=12000, sensor="DE"):
    """Helper: return dict {file_id: metadata} for 4 consecutive loads 0-3 HP."""
    return {
        start_id + k: {
            "class_name": class_name,
            "fault_size_in": fault_size,
            "fault_position": fault_position,
            "load_hp": k,
            "fs_hz": fs,
            "sensor": sensor,
        }
        for k in range(4)
    }


def build_file_id_map():
    """Return a dict {file_id: metadata_dict} for the CWRU drive-end 12 kHz data."""
    m = {}

    # ---------- Normal baseline ----------
    m.update(_make_range(97,  "Normal",    0.000, "N/A"))
    m[98] = {"class_name": "Normal", "fault_size_in": 0.000,
             "fault_position": "N/A", "load_hp": 0, "fs_hz": 12000, "sensor": "DE"}

    m.update(_make_range(105, "InnerRace", 0.007, "N/A"))
    m.update(_make_range(169, "InnerRace", 0.014, "N/A"))
    m.update(_make_range(209, "InnerRace", 0.021, "N/A"))
    # Ball
    m.update(_make_range(118, "Ball",      0.007, "N/A"))
    m.update(_make_range(185, "Ball",      0.014, "N/A"))
    m.update(_make_range(222, "Ball",      0.021, "N/A"))
    # Outer race @ 6:00 (centered)
    m.update(_make_range(130, "OuterRace", 0.007, "6:00"))
    m.update(_make_range(197, "OuterRace", 0.014, "6:00"))
    m.update(_make_range(234, "OuterRace", 0.021, "6:00"))
    # Outer race @ 3:00 (in load zone)
    m.update(_make_range(144, "OuterRace", 0.007, "3:00"))
    # Outer race @ 12:00 (orthogonal)
    m.update(_make_range(156, "OuterRace", 0.007, "12:00"))

    # ---------- NTN bearings (0.028", 0.040") ----------
    m.update(_make_range(246, "InnerRace", 0.028, "N/A"))
    m.update(_make_range(281, "InnerRace", 0.040, "N/A"))
    m.update(_make_range(258, "Ball",      0.028, "N/A"))
    m.update(_make_range(293, "Ball",      0.040, "N/A"))
    m.update(_make_range(270, "OuterRace", 0.028, "6:00"))
    m.update(_make_range(305, "OuterRace", 0.040, "6:00"))

    return m


CLASS_TO_IDX = {
    "Normal":     0,
    "Ball":       1,
    "InnerRace":  2,
    "OuterRace":  3,
}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}


def scan_cwru_dir(data_dir: str) -> pd.DataFrame:
    """
    Scan data_dir for *.mat files, parse file IDs, and build the metadata table.
    Only files whose ID is in the rule-based map are included.
    """
    id_map = build_file_id_map()
    rows = []
    for fname in sorted(os.listdir(data_dir)):
        if not fname.endswith(".mat"):
            continue
        m = re.match(r"^(\d+)\.mat$", fname)
        if not m:
            continue
        fid = int(m.group(1))
        if fid not in id_map:
            continue
        meta = id_map[fid]
        rows.append({
            "file_id":        fid,
            "file_name":      fname,
            "class_name":     meta["class_name"],
            "class_idx":      CLASS_TO_IDX[meta["class_name"]],
            "fault_size_in":  meta["fault_size_in"],
            "fault_position": meta["fault_position"],
            "load_hp":        meta["load_hp"],
            "sensor":         meta["sensor"],
            "fs_hz":          meta["fs_hz"],
            "record_id":      f"DE_{fid}",   
        })
    df = pd.DataFrame(rows).sort_values(
        ["class_idx", "fault_size_in", "fault_position", "load_hp"]
    ).reset_index(drop=True)
    return df


def inspect_mat_file(mat_path: str) -> dict:
    """
    Inspect a .mat file and return available variable names and their shapes.
    Useful for verifying the DE_time key naming convention.
    """
    mat = loadmat(mat_path)
    out = {}
    for k, v in mat.items():
        if k.startswith("__"):
            continue
        arr = np.asarray(v)
        out[k] = arr.shape
    return out


if __name__ == "__main__":
    DATA_DIR = "./dataset/full"   # <-- change to your local path
    df = scan_cwru_dir(DATA_DIR)
    pd.set_option("display.max_rows", 200)
    print(df.to_string(index=False))
    print(f"\nTotal records found: {len(df)}")
    print("\nRecords per class:")
    print(df.groupby("class_name").size())
    print("\nRecords per (class, fault_size, load):")
    print(df.groupby(["class_name", "fault_size_in", "load_hp"]).size())

    # Inspect one file to confirm variable naming
    if len(df) > 0:
        example = df.iloc[0]
        path = os.path.join(DATA_DIR, example["file_name"])
        print(f"\nVariables in {example['file_name']}:")
        for k, shape in inspect_mat_file(path).items():
            print(f"  {k}: {shape}")

 file_id file_name class_name  class_idx  fault_size_in fault_position  load_hp sensor  fs_hz record_id
      97    97.mat     Normal          0          0.000            N/A        0     DE  12000     DE_97
      98    98.mat     Normal          0          0.000            N/A        0     DE  12000     DE_98
      99    99.mat     Normal          0          0.000            N/A        2     DE  12000     DE_99
     100   100.mat     Normal          0          0.000            N/A        3     DE  12000    DE_100
     118   118.mat       Ball          1          0.007            N/A        0     DE  12000    DE_118
     119   119.mat       Ball          1          0.007            N/A        1     DE  12000    DE_119
     120   120.mat       Ball          1          0.007            N/A        2     DE  12000    DE_120
     121   121.mat       Ball          1          0.007            N/A        3     DE  12000    DE_121
     185   185.mat       Ball          1          0.014         

2

In [ ]:


import os
import numpy as np
import pandas as pd
from scipy.io import loadmat
from dataclasses import dataclass
from typing import Dict, List, Tuple


@dataclass
class SplitConfig:
    window_len: int = 1024
    stride: int = 512
    n_labeled_per_class: int = 20
    n_aux_per_class: int = 200
    test_frac_per_class: float = 0.3
    seed: int = 0
    fault_sizes: tuple = (0.007, 0.014, 0.021)   # SKF subset
    loads: tuple = (0, 1, 2, 3)
    outer_race_position: str = "6:00"


def load_de_signal(mat_path: str, file_id: int) -> np.ndarray:
    """
    Load the drive-end time series from a CWRU .mat file.
    Tries several key conventions.
    """
    mat = loadmat(mat_path)
    candidates = [
        f"X{file_id:03d}_DE_time",
        f"X{file_id}_DE_time",
        f"X{file_id:03d}_DE_time".lower(),
    ]
    for key in candidates:
        if key in mat:
            return np.asarray(mat[key]).ravel().astype(np.float64)
    for k in mat.keys():
        if k.endswith("_DE_time"):
            return np.asarray(mat[k]).ravel().astype(np.float64)
    raise KeyError(f"No DE_time variable in {mat_path}. Keys: {list(mat.keys())}")


def make_windows(signal: np.ndarray, window_len: int, stride: int) -> np.ndarray:
    if len(signal) < window_len:
        return np.empty((0, window_len))
    n = 1 + (len(signal) - window_len) // stride
    idx = np.arange(n)[:, None] * stride + np.arange(window_len)[None, :]
    return signal[idx]


def filter_table(df: pd.DataFrame, cfg: SplitConfig) -> pd.DataFrame:
    """Restrict the metadata table according to the config."""
    mask = (
        df["fault_size_in"].isin(cfg.fault_sizes) &
        df["load_hp"].isin(cfg.loads)
    )
    if cfg.outer_race_position is not None:
        mask &= (
            (df["class_name"] != "OuterRace") |
            (df["fault_position"] == cfg.outer_race_position)
        )
    return df[mask].reset_index(drop=True)


def split_records(df: pd.DataFrame, test_frac_per_class: float, seed: int):
    rng = np.random.default_rng(seed)
    train_ids, test_ids = [], []
    for cls_idx, group in df.groupby("class_idx"):
        recs = group["record_id"].tolist()
        rng.shuffle(recs)
        n_test = max(1, int(round(len(recs) * test_frac_per_class)))
        test_ids.extend(recs[:n_test])
        train_ids.extend(recs[n_test:])
    return sorted(train_ids), sorted(test_ids)


@dataclass
class DatasetSplit:
    X_labeled: np.ndarray
    y_labeled: np.ndarray
    X_aux:     np.ndarray
    X_test:    np.ndarray
    y_test:    np.ndarray
    train_record_ids: List[str]
    test_record_ids:  List[str]
    df_train: pd.DataFrame
    df_test:  pd.DataFrame


def build_split(df: pd.DataFrame, data_dir: str, cfg: SplitConfig) -> DatasetSplit:
    df = filter_table(df, cfg)
    rng = np.random.default_rng(cfg.seed)
    train_ids, test_ids = split_records(df, cfg.test_frac_per_class, cfg.seed)

    train_windows_by_class: Dict[int, List[np.ndarray]] = {c: [] for c in range(4)}
    for _, row in df[df["record_id"].isin(train_ids)].iterrows():
        path = os.path.join(data_dir, row["file_name"])
        sig = load_de_signal(path, row["file_id"])
        wins = make_windows(sig, cfg.window_len, cfg.stride)
        train_windows_by_class[row["class_idx"]].append(wins)

    X_lab, y_lab = [], []
    for c in range(4):
        pool = np.concatenate(train_windows_by_class[c], axis=0) \
               if train_windows_by_class[c] else np.empty((0, cfg.window_len))
        if len(pool) < cfg.n_labeled_per_class:
            raise ValueError(
                f"Not enough windows for class {IDX_TO_CLASS[c]}: {len(pool)} "
                f"(need {cfg.n_labeled_per_class})"
            )
        idx = rng.choice(len(pool), size=cfg.n_labeled_per_class, replace=False)
        X_lab.append(pool[idx])
        y_lab.append(np.full(cfg.n_labeled_per_class, c, dtype=np.int64))
    X_labeled = np.concatenate(X_lab, axis=0)
    y_labeled = np.concatenate(y_lab, axis=0)

    X_aux = []
    for c in range(4):
        pool = np.concatenate(train_windows_by_class[c], axis=0) \
               if train_windows_by_class[c] else np.empty((0, cfg.window_len))
        n = min(cfg.n_aux_per_class, len(pool))
        idx = rng.choice(len(pool), size=n, replace=False)
        X_aux.append(pool[idx])
    X_aux = np.concatenate(X_aux, axis=0)

    X_te, y_te = [], []
    for _, row in df[df["record_id"].isin(test_ids)].iterrows():
        path = os.path.join(data_dir, row["file_name"])
        sig = load_de_signal(path, row["file_id"])
        wins = make_windows(sig, cfg.window_len, cfg.stride)
        X_te.append(wins)
        y_te.append(np.full(len(wins), row["class_idx"], dtype=np.int64))
    X_test = np.concatenate(X_te, axis=0)
    y_test = np.concatenate(y_te, axis=0)

    return DatasetSplit(
        X_labeled=X_labeled, y_labeled=y_labeled,
        X_aux=X_aux, X_test=X_test, y_test=y_test,
        train_record_ids=train_ids, test_record_ids=test_ids,
        df_train=df[df["record_id"].isin(train_ids)].reset_index(drop=True),
        df_test=df[df["record_id"].isin(test_ids)].reset_index(drop=True),
    )

DATA_DIR = "./dataset/full"
df = scan_cwru_dir(DATA_DIR)
cfg = SplitConfig(seed=0)
split = build_split(df, DATA_DIR, cfg)


In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt


def plot_record(
    data_dir: str,
    row,
    n_samples: int = 4096,
    ax_time=None,
    ax_spec=None,
):
    path = os.path.join(data_dir, row["file_name"])
    sig = load_de_signal(path, row["file_id"])
    x = sig[:n_samples]
    fs = row["fs_hz"]
    t = np.arange(len(x)) / fs

    title = (f"{row['class_name']} | file {row['file_id']} | "
             f"{row['fault_size_in']:.3f}\" | {row['fault_position']} | "
             f"{row['load_hp']} HP")

    if ax_time is not None:
        ax_time.plot(t, x, lw=0.5)
        ax_time.set_title(title + " | time")
        ax_time.set_xlabel("time [s]")
        ax_time.set_ylabel("amplitude")

    if ax_spec is not None:
        X = np.fft.rfft(x)
        freqs = np.fft.rfftfreq(len(x), d=1.0 / fs)
        psd = (np.abs(X) ** 2) / len(x)
        ax_spec.plot(freqs, 10 * np.log10(psd + 1e-12), lw=0.5)
        ax_spec.set_title(title + " | power spectrum")
        ax_spec.set_xlabel("frequency [Hz]")
        ax_spec.set_ylabel("power [dB]")
        ax_spec.set_xlim(0, fs / 2)


def plot_per_class(
    df,
    data_dir: str,
    n_per_class: int = 3,
    n_samples: int = 4096,
    save_path: str = None,
):
    classes = sorted(df["class_idx"].unique())
    fig, axes = plt.subplots(
        len(classes) * n_per_class, 2,
        figsize=(14, 3 * len(classes) * n_per_class),
    )
    row_idx = 0
    for c in classes:
        subset = df[df["class_idx"] == c].head(n_per_class)
        for _, row in subset.iterrows():
            plot_record(data_dir, row, n_samples=n_samples,
                        ax_time=axes[row_idx, 0],
                        ax_spec=axes[row_idx, 1])
            row_idx += 1
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved {save_path}")
    plt.show()


DATA_DIR = "./dataset/full"
df = scan_cwru_dir(DATA_DIR)
cfg = SplitConfig(seed=0)
split = build_split(df, DATA_DIR, cfg)

# Plot 3 training records per class
plot_per_class(
    split.df_train, DATA_DIR, n_per_class=3,
    save_path="fig_train_time_spectrum.png",
)

# Plot 1 test record per class (held-out)
plot_per_class(
    split.df_test, DATA_DIR, n_per_class=1,
    save_path="fig_test_time_spectrum.png",
)